# 03. Advanced Hybrid Pipeline (Non-Linear)
**Objective:** Improve active classification accuracy by fusing unsupervised Dictionary Learning sparse codes with physical Time-Domain energy features, using a non-linear Random Forest Classifier.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
import os
import h5py
import numpy as np
import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

sys.path.append(os.path.abspath('../'))
from src.config import PREPROCESSED_DIR, MODELS_DIR, MOVEMENT_LABELS
from src.features import extract_fft_magnitude, EMGDictionaryLearner, extract_time_domain_features

### 1. Load the Dense Dataset

In [2]:
data_path = os.path.join(PREPROCESSED_DIR, "DB1_S1_Dense.h5")
with h5py.File(data_path, 'r') as f:
    X_bal = np.array(f['X'])
    y_bal = np.array(f['y']).astype(np.int64)
    reps_bal = np.array(f['reps'])

# Standard NinaPro DB1 Train/Test Split
train_reps = [1, 3, 4, 6, 7, 8, 9, 10]
test_reps = [2, 5]

train_idx = np.where(np.isin(reps_bal, train_reps))[0]
test_idx = np.where(np.isin(reps_bal, test_reps))[0]

X_train, y_train = X_bal[train_idx], y_bal[train_idx]
X_test, y_test = X_bal[test_idx], y_bal[test_idx]

print(f"Training Windows: {X_train.shape[0]}")
print(f"Testing Windows: {X_test.shape[0]}")

Training Windows: 143956
Testing Windows: 37562


### 2. Feature Extraction (Hybrid Fusion)

In [3]:
# A. Dictionary Learning (Frequency Domain)
X_train_freq = extract_fft_magnitude(X_train)
X_test_freq = extract_fft_magnitude(X_test)

print("\nLoading previously trained FFT Dictionary...")
dl_model = EMGDictionaryLearner(n_atoms=128, n_nonzero_coefs=5)
dl_model.load(os.path.join(MODELS_DIR, "fft_dictionary_s1.pkl"))

print("Extracting Sparse Codes...")
X_train_sparse = dl_model.transform(X_train_freq)
X_test_sparse = dl_model.transform(X_test_freq)

# B. Energy Extraction (Time Domain)
X_train_td = extract_time_domain_features(X_train)
X_test_td = extract_time_domain_features(X_test)

# C. Feature Fusion
X_train_hybrid = np.hstack((X_train_sparse, X_train_td))
X_test_hybrid = np.hstack((X_test_sparse, X_test_td))

print(f"\nFinal Hybrid Feature Space Shape: {X_train_hybrid.shape}")
print(f"(128 DL Atoms + 20 MAV/RMS Features)")

Extracting FFT magnitudes...
Extracting FFT magnitudes...

Loading previously trained FFT Dictionary...
Extracting Sparse Codes...
Extracting Time-Domain features (MAV, RMS)...
Extracting Time-Domain features (MAV, RMS)...

Final Hybrid Feature Space Shape: (143956, 148)
(128 DL Atoms + 20 MAV/RMS Features)


### 3. Train Non-Linear Classifier (Random Forest)

In [4]:
# Standardize the hybrid features
scaler_hybrid = StandardScaler()
X_train_scaled = scaler_hybrid.fit_transform(X_train_hybrid)
X_test_scaled = scaler_hybrid.transform(X_test_hybrid)

# Save new scaler
joblib.dump(scaler_hybrid, os.path.join(MODELS_DIR, "scaler_hybrid_s1.pkl"))

print("\nTraining Random Forest Classifier (100 Trees)...")
# Random Forest is highly parallelized and handles non-linear boundaries natively
rf_classifier = RandomForestClassifier(n_estimators=100, max_depth=20, n_jobs=-1, random_state=42)
rf_classifier.fit(X_train_scaled, y_train)

# Save the RF classifier
joblib.dump(rf_classifier, os.path.join(MODELS_DIR, "rf_classifier_hybrid_s1.pkl"))


Training Random Forest Classifier (100 Trees)...


['/workspaces/TCC/models/rf_classifier_hybrid_s1.pkl']

### 4. Evaluate Performance

In [5]:
preds = rf_classifier.predict(X_test_scaled)

overall_acc = accuracy_score(y_test, preds)

# Calculate Active Blocks Accuracy (Ignoring Rest)
active_idx = y_test > 0
active_acc = accuracy_score(y_test[active_idx], preds[active_idx])

print(f"\n--- SOTA Hybrid Pipeline Results ---")
print(f"Overall Accuracy:       {overall_acc * 100:.2f}%")
print(f"Active Blocks Accuracy: {active_acc * 100:.2f}%")


--- SOTA Hybrid Pipeline Results ---
Overall Accuracy:       60.04%
Active Blocks Accuracy: 60.04%
